In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = True  # set False for CPU-only notebooks

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available; running on CPU.")


CPU threads set to: 12
CUDA device: NVIDIA GeForce RTX 5070 Ti


# Kvasir-VQA x1 — Fusion (TF-IDF + ViT CLS) with MLP

This notebook keeps the same feature construction as the logistic-regression fusion baseline (concatenate TF-IDF question features + frozen ViT CLS embeddings), but replaces the classifier with a small MLP and early stopping.

Outputs are stored under `2_modeling/03_fusion/out/02_mlp/` following the existing x1 conventions.


In [2]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, ViTModel

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


2026-01-29 23:44:01.235600: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-29 23:44:01.235638: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-29 23:44:01.236628: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-29 23:44:01.242980: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-29 23:44:02.203014: W tensorflow/compiler/tf2

In [3]:
# Paths & config

def find_kvasir_x1_root() -> Path:
    import os
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError(
        "Could not locate Kvasir_VQA_x1 dataset root. \n"
        "Run this notebook from within the Kvasir_VQA_x1 folder, \n"
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "03_fusion" / "out" / "02_mlp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 32
NUM_WORKERS = 0
TOP_K = 200

MAX_FEATURES = 5000
NGRAM_RANGE = (1, 2)
TEXT_DIM = 256  # keep aligned with fusion baseline

HIDDEN_SIZES = [512, 256]  # two-layer MLP
DROPOUT = 0.3
LR = 1e-3
EPOCHS = 40
PATIENCE = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
MLP_DTYPE = torch.float32

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)
print("Device:", DEVICE)
print("Torch dtype:", TORCH_DTYPE)
print("MLP dtype:", MLP_DTYPE)



Data root: /home/aristotle/Documents/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Documents/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Documents/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/03_fusion/out/02_mlp
Device: cuda
Torch dtype: torch.float16
MLP dtype: torch.float32


In [4]:
# Load metadata and splits
meta = pd.read_csv(META_CSV)

images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

meta["question_norm"] = meta["question"].fillna("").astype(str).str.lower().str.strip()
meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

if "split" not in meta.columns:
    raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

train_df = meta[meta["split"] == "train"].reset_index(drop=True)
val_df = meta[meta["split"] == "validation"].reset_index(drop=True)
test_df = meta[meta["split"] == "test"].reset_index(drop=True)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})



{'train': 143594, 'val': 0, 'test': 15955}


In [5]:
# Top-K answers
answer_counts = train_df["answer_norm"].value_counts()
TOP_K_ANSWERS = answer_counts.head(TOP_K).index.tolist()

train_k = train_df[train_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
val_k = val_df[val_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)
test_k = test_df[test_df["answer_norm"].isin(TOP_K_ANSWERS)].reset_index(drop=True)

print("Top-K answers:", len(TOP_K_ANSWERS))
print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})



Top-K answers: 200
{'train': 38424, 'val': 0, 'test': 4252}


In [6]:
# Helper utilities

def compute_metrics(y_true, y_pred):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
    }
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return metrics, report


def save_metrics(prefix, split_name, metrics, report, extra=None):
    payload = {"metrics": metrics, "report": report}
    if extra is not None:
        payload["extra"] = extra
    with open(OUT_DIR / f"{prefix}_metrics_{split_name}.json", "w") as f:
        json.dump(payload, f, indent=2)


def save_predictions(prefix, split_name, df, y_pred):
    cols = [c for c in ["img_id", "question", "answer", "answer_norm"] if c in df.columns]
    pred_df = df[cols].copy()
    pred_df["pred"] = y_pred
    pred_df.to_csv(OUT_DIR / f"{prefix}_pred_{split_name}.csv", index=False)


def save_probs(prefix, split_name, probs, classes, ids):
    np.savez(
        OUT_DIR / f"{prefix}_probs_{split_name}.npz",
        probs=probs,
        classes=np.array(classes),
        img_ids=np.array(ids),
    )


def align_probs(probs, from_classes, target_classes):
    idx = {c: i for i, c in enumerate(from_classes)}
    return np.stack([probs[:, idx[c]] for c in target_classes], axis=1)



In [7]:
# Text features (TF-IDF, optional SVD like fusion baseline)
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=NGRAM_RANGE)
X_text_train_sparse = vectorizer.fit_transform(train_k["question_norm"])
X_text_val_sparse = vectorizer.transform(val_k["question_norm"]) if len(val_k) else None
X_text_test_sparse = vectorizer.transform(test_k["question_norm"]) if len(test_k) else None

n_text_features = X_text_train_sparse.shape[1]
if n_text_features <= TEXT_DIM:
    X_text_train = X_text_train_sparse.toarray()
    X_text_val = X_text_val_sparse.toarray() if X_text_val_sparse is not None else None
    X_text_test = X_text_test_sparse.toarray() if X_text_test_sparse is not None else None
    text_dim_used = n_text_features
else:
    svd = TruncatedSVD(n_components=TEXT_DIM, random_state=SEED)
    X_text_train = svd.fit_transform(X_text_train_sparse)
    X_text_val = svd.transform(X_text_val_sparse) if X_text_val_sparse is not None else None
    X_text_test = svd.transform(X_text_test_sparse) if X_text_test_sparse is not None else None
    text_dim_used = TEXT_DIM

print("Text feature dim:", text_dim_used)



Text feature dim: 256


In [8]:
# Image embeddings (reuse cache from image-only baseline)
class ImageDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["img_id"]
        img = Image.open(row["image_path"]).convert("RGB")
        return img_id, img


def collate_fn(batch):
    ids = [b[0] for b in batch]
    images = [b[1] for b in batch]
    inputs = processor(images=images, return_tensors="pt")
    return ids, inputs["pixel_values"]


def compute_embeddings(unique_df):
    ds = ImageDS(unique_df)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
    emb_map = {}
    with torch.no_grad():
        for ids, pixels in tqdm(dl, desc="ViT embed"):
            pixels = pixels.to(DEVICE)
            out = vit(pixels).last_hidden_state[:, 0, :]
            for i, img_id in enumerate(ids):
                emb_map[img_id] = out[i].cpu().numpy()
    return emb_map


processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
try:
    vit = ViTModel.from_pretrained(MODEL_NAME, torch_dtype=TORCH_DTYPE)
    vit = vit.to(DEVICE)
except RuntimeError as e:
    print(f"Falling back to CPU for ViT (error: {e}).")
    DEVICE = torch.device("cpu")
    TORCH_DTYPE = torch.float32
    vit = ViTModel.from_pretrained(MODEL_NAME, torch_dtype=TORCH_DTYPE).to(DEVICE)
vit.eval()

EMB_PATH = DATA_ROOT / "2_modeling" / "02_image_only" / "out" / "image_embeddings.npz"
unique_imgs = pd.concat([train_k, val_k, test_k])[ ["img_id", "image_path"] ].drop_duplicates()

if EMB_PATH.exists():
    data = np.load(EMB_PATH, allow_pickle=True)
    img_ids = data["img_ids"].tolist()
    embeddings = data["embeddings"]
    emb_map = {img_id: embeddings[i] for i, img_id in enumerate(img_ids)}
    print("Loaded cached embeddings:", len(emb_map))
else:
    emb_map = compute_embeddings(unique_imgs)
    img_ids = list(emb_map.keys())
    embeddings = np.stack([emb_map[i] for i in img_ids])
    EMB_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.savez(EMB_PATH, img_ids=np.array(img_ids), embeddings=embeddings)
    print("Saved embeddings:", EMB_PATH)


def build_X_img(df):
    return np.stack([emb_map[i] for i in df["img_id"].tolist()])

X_img_train = build_X_img(train_k)
X_img_val = build_X_img(val_k) if len(val_k) else None
X_img_test = build_X_img(test_k) if len(test_k) else None

img_scaler = StandardScaler()
X_img_train = img_scaler.fit_transform(X_img_train)
if X_img_val is not None:
    X_img_val = img_scaler.transform(X_img_val)
if X_img_test is not None:
    X_img_test = img_scaler.transform(X_img_test)

print("Image feature dim:", X_img_train.shape[1])



Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loaded cached embeddings: 6329
Image feature dim: 768


In [9]:
# Concatenate features
X_train = np.hstack([X_img_train, X_text_train]).astype(np.float32)
X_val = np.hstack([X_img_val, X_text_val]).astype(np.float32) if X_img_val is not None else None
X_test = np.hstack([X_img_test, X_text_test]).astype(np.float32) if X_img_test is not None else None

n_features = X_train.shape[1]
classes = sorted(TOP_K_ANSWERS)
class_to_idx = {c: i for i, c in enumerate(classes)}

y_train = train_k["answer_norm"].map(class_to_idx).values

y_val = val_k["answer_norm"].map(class_to_idx).values if len(val_k) else None
y_test = test_k["answer_norm"].map(class_to_idx).values if len(test_k) else None

print("Feature dim:", n_features, "#classes:", len(classes))



Feature dim: 1024 #classes: 200


In [10]:
# Torch dataset
class NPZDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.as_tensor(X, dtype=MLP_DTYPE)
        self.y = None if y is None else torch.from_numpy(y).long()

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is None:
            return self.X[idx]
        return self.X[idx], self.y[idx]



In [11]:
# MLP model
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, num_classes, dropout=0.3):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0
    for Xb, yb in loader:
        Xb = Xb.to(DEVICE, dtype=MLP_DTYPE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(Xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader):
    model.eval()
    ys = []
    ps = []
    with torch.no_grad():
        for Xb, yb in loader:
            Xb = Xb.to(DEVICE, dtype=MLP_DTYPE)
            logits = model(Xb)
            probs = torch.softmax(logits, dim=1)
            ps.append(probs.cpu())
            ys.append(yb)
    y_true = torch.cat(ys).numpy()
    probs_all = torch.cat(ps).numpy()
    y_pred = probs_all.argmax(axis=1)
    return y_true, y_pred, probs_all



In [12]:
# DataLoaders
train_ds = NPZDataset(X_train, y_train)
val_ds = NPZDataset(X_val, y_val) if X_val is not None else None
test_ds = NPZDataset(X_test, y_test) if X_test is not None else None

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS) if val_ds else None
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS) if test_ds else None



In [13]:
# Training loop with early stopping on val macro-F1
model = MLP(n_features, HIDDEN_SIZES, len(classes), dropout=DROPOUT).to(DEVICE, dtype=MLP_DTYPE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_state = None
best_val_f1 = -1
patience = PATIENCE
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer)
    if val_loader is not None:
        y_true, y_pred, probs = eval_epoch(model, val_loader)
        # compute f1 on CPU
        val_metrics, _ = compute_metrics(y_true, y_pred)
        val_f1 = val_metrics["macro_f1"]
        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})
        print(f"Epoch {epoch}: train_loss={train_loss:.4f} val_acc={val_metrics['accuracy']:.4f} val_f1={val_f1:.4f}")
        if val_f1 > best_val_f1 + 1e-4:
            best_val_f1 = val_f1
            best_state = model.state_dict()
            patience = PATIENCE
        else:
            patience -= 1
            if patience == 0:
                print("Early stopping")
                break
    else:
        history.append({"epoch": epoch, "train_loss": train_loss})

if best_state is not None:
    model.load_state_dict(best_state)



In [14]:
# Evaluate splits and save artifacts

def eval_and_save(split_name, loader, df, prefix="mlp_fusion"):
    if loader is None:
        return None
    y_true, y_pred, probs = eval_epoch(model, loader)
    # map back to labels
    y_true_lbl = [classes[i] for i in y_true]
    y_pred_lbl = [classes[i] for i in y_pred]
    metrics, report = compute_metrics(y_true_lbl, y_pred_lbl)
    save_metrics(prefix, split_name, metrics, report)
    save_predictions(prefix, split_name, df, y_pred_lbl)
    save_probs(prefix, split_name, probs, classes, df["img_id"].values)
    print(split_name, metrics)
    return metrics

train_metrics = eval_and_save("train", train_loader, train_k)
val_metrics = eval_and_save("val", val_loader, val_k)
test_metrics = eval_and_save("test", test_loader, test_k)

# Save training history
pd.DataFrame(history).to_csv(OUT_DIR / "training_history.csv", index=False)

# Save model weights
model_path = OUT_DIR / "mlp_fusion.pt"
torch.save({
    "state_dict": model.state_dict(),
    "classes": classes,
    "config": {
        "input_dim": n_features,
        "hidden_sizes": HIDDEN_SIZES,
        "dropout": DROPOUT,
    },
    "vectorizer": vectorizer,
    "svd": svd if 'svd' in globals() else None,
    "img_scaler": img_scaler,
}, model_path)
print("Saved model to", model_path)



train {'accuracy': 0.49554965646470955, 'macro_f1': 0.25983165753048026}
test {'accuracy': 0.4409689557855127, 'macro_f1': 0.19533447090969708}
Saved model to /home/aristotle/Documents/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/03_fusion/out/02_mlp/mlp_fusion.pt


In [15]:
# Summary table
summary_rows = []
for name, mets in [("train", train_metrics), ("val", val_metrics), ("test", test_metrics)]:
    if mets:
        summary_rows.append({"split": name, "accuracy": mets["accuracy"], "macro_f1": mets["macro_f1"]})

if summary_rows:
    display(pd.DataFrame(summary_rows).set_index("split").round(4))
else:
    print("No metrics collected.")



,accuracy,macro_f1
split,,
train,0.4955,0.2598
test,0.4410,0.1953
